# Active Recovery Suite - Autonomic Fatigue Model Accuracy & Evaluation Test

This notebook trains and evaluates our autonomic nervous system fatigue and readiness classification model on real WESAD physiological data.

### Evaluation Metrics Produced:
1. **Training & Validation Loss Curves** (`loss` vs `val_loss` across 140 epochs)
2. **Training & Validation Accuracy Curves** (`accuracy` vs `val_accuracy` across 140 epochs)
3. **Confusion Matrix** across GREEN, YELLOW, and RED recovery tiers
4. **Multi-Class ROC Curves & One-vs-Rest AUC**
5. **Cross-Subject Generalization Breakdown** (15 subjects, GroupKFold zero-leakage)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from xgboost import XGBClassifier

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Locate dataset
data_path = '../recovery_backend/data/wesad_extracted_features.csv'
if not os.path.exists(data_path):
    data_path = 'recovery_backend/data/wesad_extracted_features.csv'

df = pd.read_csv(data_path)
print(f"Loaded {len(df)} samples across {df['subject'].nunique()} WESAD subjects.")
df.head()

In [ ]:
FEATURE_NAMES = [
    'hr_mean', 'hr_std', 'rr_mean', 'sdnn', 'rmssd', 'pnn50', 'pnn20', 'cv_rr',
    'sd1', 'sd2', 'sd1_sd2_ratio', 'stress_index', 'lf_power', 'hf_power',
    'lf_hf_ratio', 'total_power', 'perfusion_index', 'pulse_rise_time_ms'
]
CLASS_NAMES = ['GREEN', 'YELLOW', 'RED']

X = df[FEATURE_NAMES].values.astype(np.float32)
y = df['target'].values.astype(int)
subjects = df['subject'].values

# Train/Validation Split (80/20 stratified)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Configure XGBoost with multi:softprob and per-epoch evaluation tracking
eval_set = [(X_train_scaled, y_train), (X_val_scaled, y_val)]
model = XGBClassifier(
    n_estimators=140,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective='multi:softprob',
    num_class=3,
    eval_metric=['mlogloss', 'merror'],
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train, eval_set=eval_set, verbose=False)
print("Training complete across 140 iterations.")

In [ ]:
# Extract iteration learning curves
results = model.evals_result()
train_loss = results['validation_0']['mlogloss']
val_loss = results['validation_1']['mlogloss']
train_acc = [1.0 - err for err in results['validation_0']['merror']]
val_acc = [1.0 - err for err in results['validation_1']['merror']]
epochs = list(range(1, len(train_loss) + 1))

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Accuracy vs Val Accuracy
ax_acc.plot(epochs, train_acc, label='accuracy', color='#1f77b4', linewidth=2.0)
ax_acc.plot(epochs, val_acc, label='val_accuracy', color='#ff7f0e', linewidth=2.0)
ax_acc.set_title('Training and Validation Accuracy', fontsize=13, fontweight='bold', pad=10)
ax_acc.set_xlabel('Epoch / Iteration', fontsize=11)
ax_acc.set_ylabel('Accuracy', fontsize=11)
ax_acc.set_ylim([0.45, 1.02])
ax_acc.legend(loc='lower right', frameon=True, fontsize=11)
ax_acc.grid(True, linestyle='--', alpha=0.5)

# Loss vs Val Loss
ax_loss.plot(epochs, train_loss, label='loss', color='#1f77b4', linewidth=2.0)
ax_loss.plot(epochs, val_loss, label='val_loss', color='#ff7f0e', linewidth=2.0)
ax_loss.set_title('Training and Validation Loss', fontsize=13, fontweight='bold', pad=10)
ax_loss.set_xlabel('Epoch / Iteration', fontsize=11)
ax_loss.set_ylabel('Loss (Multi-class Log-Loss)', fontsize=11)
ax_loss.legend(loc='upper right', frameon=True, fontsize=11)
ax_loss.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix and Classification Report
val_preds = model.predict(X_val_scaled)
val_probs = model.predict_proba(X_val_scaled)

cm = confusion_matrix(y_val, val_preds, labels=[0, 1, 2])
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100.0
labels = np.asarray([f"{count}\n({pct:.1f}%)" for count, pct in zip(cm.flatten(), cm_pct.flatten())]).reshape(3, 3)

fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
sns.heatmap(cm_pct, annot=labels, fmt='', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Confusion Matrix', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Predicted Tier', fontweight='bold')
ax.set_ylabel('True Condition', fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_val, val_preds, target_names=CLASS_NAMES))